# reduce-gather-sum — worked example 1: Global max via all_gather + local reduce

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-gather-sum`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`dist.all_gather` collects every rank's tensor into a per-rank list on EVERY rank, preserving each individual value. Once you hold the full set locally you can apply any reduction — even ones `ReduceOp` supports — but the point of gather is access to the raw per-rank values, not just the aggregate. Here we gather then take a local max.

## Worked solution

We model the distributed runtime with a small `FakeDist` so the logic is testable in a single process: it holds the list of all ranks' values and its `all_gather(out_list, tensor)` copies each rank's value into `out_list`. The function `global_max_via_gather` pre-allocates `gather_list = [zeros(1) for _ in range(world_size)]` on the calling rank, calls `all_gather` to populate it, concatenates into a flat `(world_size,)` tensor, and takes `.max()`. We return the Python float. The lesson: gather gives you every value, so you can compute the max locally — and the same gathered list could equally feed a sum, median, or any other reduction.

In [ ]:
class FakeDist:
    """Single-process stand-in: holds every rank's scalar."""
    def __init__(self, all_values):
        self.all_values = list(all_values)
    def all_gather(self, out_list, tensor):
        for i, v in enumerate(self.all_values):
            out_list[i].copy_(t.tensor([float(v)]))


def global_max_via_gather(world_size, dist_module):
    gather_list = [t.zeros(1) for _ in range(world_size)]
    dist_module.all_gather(gather_list, t.zeros(1))
    gathered = t.cat(gather_list)
    return gathered.max().item()


vals = [3.0, 7.0, 1.0, 5.0]
fd = FakeDist(vals)
print('global max:', global_max_via_gather(len(vals), fd))